In [ ]:
!pip install -q catboost

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import os

In [ ]:
# GPU 가용 여부 자동 감지
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {DEVICE}")

In [ ]:
SEED = 42
N_SPLITS = 5
TARGET = "임신 성공 여부"

np.random.seed(SEED)

In [ ]:


# ─────────────────────────────────────────
# 0. 설정
# ─────────────────────────────────────────
   # ← 실제 컬럼명으로 교체



# ─────────────────────────────────────────
# 1. 데이터 로드
# ─────────────────────────────────────────
def load_data(train_path="train.csv", test_path="test.csv"):
    train = pd.read_csv(train_path, encoding="utf-8-sig")
    test  = pd.read_csv(test_path,  encoding="utf-8-sig")
    print(f"Train: {train.shape}, Test: {test.shape}")
    print(f"Target 분포:\n{train[TARGET].value_counts(normalize=True).round(3)}")
    return train, test


# ─────────────────────────────────────────
# 2. 전처리
# ─────────────────────────────────────────

# ── 2-1. 순서형 인코딩 매핑 (도메인 지식 기반) ──
AGE_ORDER = {            # 나이 그룹 → 순서형
    "18-34": 0, "35-37": 1, "38-39": 2,
    "40-42": 3, "43-44": 4, "45-50": 5,
}

COUNT_ORDER = {          # 횟수형 컬럼 (예: 시술 횟수, 임신 횟수 등) → 순서형
    "0":    0, "1":    1, "2":    2,
    "3":    3, "4":    4, "5":    5,
    "6 이상": 6,
}

def encode_ordinal(df, col, mapping):
    """값 → 순서 정수 변환. 매핑에 없는 값은 -1."""
    return df[col].map(mapping).fillna(-1).astype(int)


def preprocess(train: pd.DataFrame, test: pd.DataFrame):
    """
    전처리 단계
    1) 나이 컬럼 → 순서형
    2) 횟수 컬럼 → 순서형
    3) 나머지 범주형 → Label Encoding
    4) 결측 처리
    """
    combined = pd.concat([train.drop(columns=[TARGET]), test], axis=0).reset_index(drop=True)
    n_train  = len(train)

    # ── 나이 컬럼 (실제 컬럼명으로 교체) ──
    AGE_COLS = [c for c in combined.columns if "나이" in c or "age" in c.lower()]
    for col in AGE_COLS:
        combined[col] = encode_ordinal(combined, col, AGE_ORDER)

    # ── 횟수 컬럼 (실제 컬럼명으로 교체) ──
    COUNT_COLS = [c for c in combined.columns
                  if any(k in c for k in ["횟수", "번", "회", "수"])]
    for col in COUNT_COLS:
        combined[col] = encode_ordinal(combined, col, COUNT_ORDER)

    # ── 이진/불임 원인 컬럼: 이미 0/1이면 수치 그대로 ──

    # ── 나머지 object 컬럼 → Label Encoding ──
    obj_cols = combined.select_dtypes(include="object").columns.tolist()
    le = LabelEncoder()
    for col in obj_cols:
        combined[col] = combined[col].astype(str)
        combined[col] = le.fit_transform(combined[col])

    # ── inf → NaN 치환 후 중앙값으로 결측 처리 ──
    combined = combined.replace([np.inf, -np.inf], np.nan)
    combined = combined.fillna(combined.median(numeric_only=True))
    combined = combined.replace([np.inf, -np.inf], 0)   # 중앙값도 nan인 극단 케이스 방어

    X_train = combined.iloc[:n_train].reset_index(drop=True)
    X_test  = combined.iloc[n_train:].reset_index(drop=True)
    y_train = train[TARGET].values

    print(f"전처리 완료 → Feature 수: {X_train.shape[1]}")
    return X_train, X_test, y_train


# ─────────────────────────────────────────
# 3. 피처 엔지니어링
# ─────────────────────────────────────────
def feature_engineering(X_train: pd.DataFrame, X_test: pd.DataFrame):
    """
    도메인 파생 피처 생성
    - 이식배아비율 = 이식된 배아 수 / (전체 배아 수 + 1)
    - 불임원인합계 = 불임원인 이진 컬럼들의 합
    - 임신출산비율 = 이전 임신 횟수 / (이전 출산 횟수 + 1)
    """
    for df in [X_train, X_test]:

        # 이식배아비율 (분모 0 방어: +1 이미 적용, 결과 inf 클리핑)
        emb_transferred = next((c for c in df.columns if "이식" in c and "배아" in c), None)
        emb_total       = next((c for c in df.columns if "전체" in c and "배아" in c), None)
        if emb_transferred and emb_total:
            df["이식배아비율"] = (df[emb_transferred] / (df[emb_total] + 1)).replace([np.inf, -np.inf], 0)

        # 불임원인합계 (컬럼명에 '불임원인' 포함)
        infertility_cols = [c for c in df.columns if "불임원인" in c]
        if infertility_cols:
            df["불임원인합계"] = df[infertility_cols].sum(axis=1)

        # 임신출산비율 (분모 0 방어)
        preg_col  = next((c for c in df.columns if "임신" in c and "횟수" in c), None)
        birth_col = next((c for c in df.columns if "출산" in c and "횟수" in c), None)
        if preg_col and birth_col:
            df["임신출산비율"] = (df[preg_col] / (df[birth_col] + 1)).replace([np.inf, -np.inf], 0)

    # 파생 피처 생성 후 남아있는 inf/nan 일괄 정리
    for df in [X_train, X_test]:
        num_cols = df.select_dtypes(include=[np.number]).columns
        df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
        df[num_cols] = df[num_cols].fillna(df[num_cols].median())

    print(f"피처 엔지니어링 완료 → Feature 수: {X_train.shape[1]}")
    return X_train, X_test


# ─────────────────────────────────────────
# 4. 개별 모델 정의
# ─────────────────────────────────────────

# ── PyTorch MLP (GPU 지원) ──
class TorchMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


class TorchMLPWrapper:
    """sklearn-like interface for PyTorch MLP"""

    def __init__(self, epochs=50, batch_size=1024, lr=1e-3, patience=5):
        self.epochs     = epochs
        self.batch_size = batch_size
        self.lr         = lr
        self.patience   = patience
        self.model      = None
        # DEVICE를 인스턴스에 저장 → 모듈 스코프 참조 오류 방지
        self.device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def _to_np32(self, X):
        """DataFrame/ndarray → float32 numpy"""
        arr = X.values if hasattr(X, "values") else X
        return np.nan_to_num(arr.astype(np.float32), nan=0.0, posinf=3.4e38, neginf=-3.4e38)

    def fit(self, X_tr, y_tr, eval_set=None, **kwargs):
        X_np = self._to_np32(X_tr)
        y_np = y_tr.astype(np.float32)

        # 클래스 불균형 가중치
        pos_w      = float((y_np == 0).sum()) / float((y_np == 1).sum() + 1e-9)
        pos_weight = torch.tensor([pos_w]).to(self.device)
        criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        self.model = TorchMLP(X_np.shape[1]).to(self.device)
        optimizer  = torch.optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=1e-4)
        scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.epochs)

        dataset = TensorDataset(torch.tensor(X_np), torch.tensor(y_np))
        loader  = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        best_loss, patience_cnt = float("inf"), 0
        best_state = None

        for epoch in range(self.epochs):
            self.model.train()
            for xb, yb in loader:
                xb, yb = xb.to(self.device), yb.to(self.device)
                optimizer.zero_grad()
                criterion(self.model(xb), yb).backward()
                optimizer.step()
            scheduler.step()

            # Early stopping (val set 기반)
            if eval_set:
                X_val_np = self._to_np32(eval_set[0][0])
                y_val_np = eval_set[0][1].astype(np.float32)
                with torch.no_grad():
                    self.model.eval()
                    xv = torch.tensor(X_val_np).to(self.device)
                    yv = torch.tensor(y_val_np).to(self.device)
                    val_loss = criterion(self.model(xv), yv).item()
                if val_loss < best_loss:
                    best_loss    = val_loss
                    best_state   = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                    patience_cnt = 0
                else:
                    patience_cnt += 1
                    if patience_cnt >= self.patience:
                        break

        if best_state:
            self.model.load_state_dict(best_state)

    def predict_proba(self, X):
        X_np = self._to_np32(X)
        self.model.eval()
        with torch.no_grad():
            logits = self.model(torch.tensor(X_np).to(self.device)).cpu().numpy()
        prob = 1 / (1 + np.exp(-logits))
        return np.column_stack([1 - prob, prob])


def get_models():
    gpu_available = torch.cuda.is_available()

    models = {
        # ── LightGBM: GPU ──
        "lgbm": lgb.LGBMClassifier(
            n_estimators=1000, learning_rate=0.05,
            num_leaves=127, max_depth=-1,
            colsample_bytree=0.8, subsample=0.8,
            reg_alpha=0.1, reg_lambda=0.1,
            class_weight="balanced",
            device="gpu" if gpu_available else "cpu",
            random_state=SEED, n_jobs=-1, verbose=-1,
        ),
        # ── XGBoost: CUDA ──
        "xgb": xgb.XGBClassifier(
            n_estimators=1000, learning_rate=0.05,
            max_depth=6, subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=3,
            eval_metric="auc",
            device="cuda" if gpu_available else "cpu",
            random_state=SEED,
            verbosity=0,
        ),
        # ── CatBoost: GPU ──
        "cat": CatBoostClassifier(
            iterations=1000, learning_rate=0.05,
            depth=6, l2_leaf_reg=3,
            auto_class_weights="Balanced",
            task_type="GPU" if gpu_available else "CPU",
            devices="0",
            random_seed=SEED, verbose=0,
        ),
        # ── RandomForest: GPU 미지원 → CPU 유지 ──
        "rf": RandomForestClassifier(
            n_estimators=500, max_depth=12,
            min_samples_leaf=10, max_features="sqrt",
            class_weight="balanced",
            random_state=SEED, n_jobs=-1,
        ),
        # ── MLP: PyTorch GPU ──
        "mlp": TorchMLPWrapper(epochs=50, batch_size=1024, lr=1e-3, patience=5),
    }

    print(f"GPU 사용 여부: {'✅ CUDA 활성화' if gpu_available else '⚠️  CPU 폴백'}")
    return models


# ─────────────────────────────────────────
# 5. OOF + 앙상블 학습
# ─────────────────────────────────────────
def sanitize(df: pd.DataFrame) -> pd.DataFrame:
    """
    pandas 2.x 포함 모든 환경에서 inf/nan 완전 제거.
    - pd.api.types.is_numeric_dtype() 로 비수치 컬럼 감지 (StringDtype 대응)
    - numpy 레벨에서 inf/nan 직접 치환
    """
    from sklearn.preprocessing import LabelEncoder
    df = df.copy()
    cols = df.columns.tolist()
    _le = LabelEncoder()

    # 1) 비수치 컬럼 → LabelEncoding → int
    for col in cols:
        if not pd.api.types.is_numeric_dtype(df[col]):
            try:
                df[col] = _le.fit_transform(df[col].astype(str).fillna("missing"))
            except Exception:
                df[col] = 0

    # 2) 전체를 float64 numpy 배열로 변환
    try:
        arr = df.values.astype(np.float64)
    except Exception:
        # 변환 실패 시 컬럼별 강제 처리
        for col in cols:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
        arr = df.values.astype(np.float64)

    # 3) inf → 유한 최대값, nan → 컬럼 중앙값(없으면 0)
    arr = np.where(np.isposinf(arr),  3.4e38, arr)
    arr = np.where(np.isneginf(arr), -3.4e38, arr)
    for j in range(arr.shape[1]):
        mask = np.isnan(arr[:, j])
        if mask.any():
            med = np.nanmedian(arr[:, j])
            arr[mask, j] = 0.0 if np.isnan(med) else med

    # 4) 최종 안전망
    arr = np.nan_to_num(arr, nan=0.0, posinf=3.4e38, neginf=-3.4e38)

    return pd.DataFrame(arr, columns=cols, index=df.index)


def train_oof(X_train, X_test, y_train, models):
    """
    5-Fold Stratified CV로 각 모델의 OOF 예측값과 Test 예측값 생성
    Returns:
        oof_preds  : (n_train, n_models) OOF 예측 확률
        test_preds : (n_test,  n_models) 테스트 예측 확률
        oof_scores : {model_name: auc_score}
    """
    # ── 모델 진입 전 inf/nan 완전 제거 ──
    X_train = sanitize(X_train.copy())
    X_test  = sanitize(X_test.copy())

    inf_tr = np.isinf(X_train.values).sum()
    inf_te = np.isinf(X_test.values).sum()
    nan_tr = np.isnan(X_train.values).sum()
    if inf_tr + inf_te + nan_tr > 0:
        raise ValueError(f"sanitize 후에도 inf/nan 잔존: train_inf={inf_tr}, test_inf={inf_te}, train_nan={nan_tr}")
    print(f"데이터 검증 완료 (inf=0, nan=0) | train={X_train.shape}, test={X_test.shape}")

    n_train  = X_train.shape[0]
    n_test   = X_test.shape[0]
    n_models = len(models)

    oof_preds  = np.zeros((n_train, n_models))
    test_preds = np.zeros((n_test,  n_models))
    oof_scores = {}

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    for m_idx, (name, model) in enumerate(models.items()):
        print(f"\n{'='*50}")
        print(f"[{m_idx+1}/{n_models}] 모델: {name.upper()}")
        fold_scores = []
        fold_test   = np.zeros((n_test, N_SPLITS))

        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
            X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train[tr_idx],      y_train[val_idx]

            # ── 1) fold 슬라이스 후 NaN/inf 즉시 제거 (모든 모델 공통) ──
            def _clean(df):
                arr = df.values if hasattr(df, "values") else np.array(df)
                return np.nan_to_num(arr.astype(np.float64), nan=0.0, posinf=3.4e38, neginf=-3.4e38)

            X_tr_c   = pd.DataFrame(_clean(X_tr),   columns=X_tr.columns).reset_index(drop=True)
            X_val_c  = pd.DataFrame(_clean(X_val),  columns=X_val.columns).reset_index(drop=True)
            X_test_c = pd.DataFrame(_clean(X_test), columns=X_test.columns).reset_index(drop=True)

            # ── 2) XGBoost: DMatrix 직접 경로 ──
            if name == "xgb":
                dtrain = xgb.DMatrix(X_tr_c.values,   label=y_tr)
                dval   = xgb.DMatrix(X_val_c.values,  label=y_val)
                dtest  = xgb.DMatrix(X_test_c.values)
                xgb_params = dict(
                    eta=0.05, max_depth=6, subsample=0.8,
                    colsample_bytree=0.8, scale_pos_weight=3,
                    eval_metric="auc", objective="binary:logistic",
                    device="cuda" if torch.cuda.is_available() else "cpu",
                    seed=SEED, verbosity=0,
                )
                bst = xgb.train(
                    xgb_params, dtrain,
                    num_boost_round=1000,
                    evals=[(dval, "val")],
                    early_stopping_rounds=50,
                    verbose_eval=False,
                )
                best_iter = bst.best_iteration if bst.best_iteration is not None else bst.num_boosted_rounds()
                oof_preds[val_idx, m_idx] = bst.predict(dval,  iteration_range=(0, best_iter + 1))
                fold_test[:, fold]        = bst.predict(dtest, iteration_range=(0, best_iter + 1))
                auc = roc_auc_score(y_val, oof_preds[val_idx, m_idx])
                fold_scores.append(auc)
                print(f"  Fold {fold+1}: AUC = {auc:.5f}")
                continue

            # ── 3) 나머지 모델: fit_params를 clean 데이터 기준으로 구성 ──
            fit_params = {}
            if name == "lgbm":
                fit_params = dict(
                    eval_set=[(X_val_c, y_val)],
                    callbacks=[lgb.early_stopping(50, verbose=False),
                               lgb.log_evaluation(-1)],
                )
            elif name == "cat":
                fit_params = dict(
                    eval_set=(X_val_c, y_val),
                    early_stopping_rounds=50,
                )
            elif name == "mlp":
                fit_params = dict(eval_set=[(X_val_c, y_val)])

            model.fit(X_tr_c, y_tr, **fit_params)

            val_pred  = model.predict_proba(X_val_c)[:, 1]
            test_pred = model.predict_proba(X_test_c)[:, 1]

            oof_preds[val_idx, m_idx] = val_pred
            fold_test[:, fold]        = test_pred

            auc = roc_auc_score(y_val, val_pred)
            fold_scores.append(auc)
            print(f"  Fold {fold+1}: AUC = {auc:.5f}")

        test_preds[:, m_idx] = fold_test.mean(axis=1)
        mean_auc = np.mean(fold_scores)
        oof_scores[name] = mean_auc
        print(f"  → {name.upper()} Mean AUC: {mean_auc:.5f}")

    return oof_preds, test_preds, oof_scores


# ─────────────────────────────────────────
# 6. 앙상블
# ─────────────────────────────────────────
def ensemble(oof_preds, test_preds, y_train, oof_scores):
    """
    3가지 앙상블 전략:
    1) Soft Voting (단순 평균)
    2) Weighted Voting (AUC 기반 가중 평균)
    3) Stacking (LR 메타 모델)
    """
    results = {}

    # ── 1) Soft Voting ──
    soft_oof  = oof_preds.mean(axis=1)
    soft_test = test_preds.mean(axis=1)
    soft_auc  = roc_auc_score(y_train, soft_oof)
    results["soft_voting"] = {"oof": soft_oof, "test": soft_test, "auc": soft_auc}
    print(f"\n[Soft Voting]    OOF AUC: {soft_auc:.5f}")

    # ── 2) Weighted Voting ──
    scores = np.array(list(oof_scores.values()))
    weights = scores / scores.sum()
    w_oof   = (oof_preds  * weights).sum(axis=1)
    w_test  = (test_preds * weights).sum(axis=1)
    w_auc   = roc_auc_score(y_train, w_oof)
    results["weighted_voting"] = {"oof": w_oof, "test": w_test, "auc": w_auc,
                                   "weights": dict(zip(oof_scores.keys(), weights.round(4)))}
    print(f"[Weighted Voting] OOF AUC: {w_auc:.5f}  (weights={results['weighted_voting']['weights']})")

    # ── 3) Stacking (LR 메타 모델) ──
    meta = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    meta.fit(oof_preds, y_train)
    stack_oof  = meta.predict_proba(oof_preds)[:, 1]
    stack_test = meta.predict_proba(test_preds)[:, 1]
    stack_auc  = roc_auc_score(y_train, stack_oof)
    results["stacking"] = {"oof": stack_oof, "test": stack_test, "auc": stack_auc}
    print(f"[Stacking]       OOF AUC: {stack_auc:.5f}")

    # ── Best 선택 ──
    best_key = max(results, key=lambda k: results[k]["auc"])
    print(f"\n★ Best Ensemble: {best_key.upper()} (AUC={results[best_key]['auc']:.5f})")
    return results, best_key


# ─────────────────────────────────────────
# 7. 제출 파일 저장
# ─────────────────────────────────────────
def save_submission(test_pred, test_ids, best_key, results, output_dir="outputs"):
    os.makedirs(output_dir, exist_ok=True)

    # Best 모델 submission
    sub = pd.DataFrame({"ID": test_ids, "probability": test_pred})
    sub_path = f"{output_dir}/submission_best_{best_key}.csv"
    sub.to_csv(sub_path, index=False, encoding="utf-8-sig")
    print(f"\n제출 파일 저장: {sub_path}")

    # 모든 앙상블 결과 저장
    for key, val in results.items():
        path = f"{output_dir}/submission_{key}.csv"
        pd.DataFrame({"ID": test_ids, "probability": val["test"]}).to_csv(
            path, index=False, encoding="utf-8-sig")

    # AUC 요약
    summary = pd.DataFrame([
        {"strategy": k, "oof_auc": v["auc"]} for k, v in results.items()
    ]).sort_values("oof_auc", ascending=False)
    summary.to_csv(f"{output_dir}/auc_summary.csv", index=False)
    print("\n─── AUC 요약 ───")
    print(summary.to_string(index=False))


# ─────────────────────────────────────────
# 8. 메인 실행
# ─────────────────────────────────────────
def main():
    print("=" * 60)
    print("   난임 환자 임신 성공 예측 AI 파이프라인")
    print("=" * 60)

    # 1. 데이터 로드
    train, test = load_data()

    # 2. 전처리
    X_train, X_test, y_train = preprocess(train, test)

    # 3. 피처 엔지니어링
    X_train, X_test = feature_engineering(X_train, X_test)

    # 4. 모델 정의
    models = get_models()

    # 5. OOF 학습
    oof_preds, test_preds, oof_scores = train_oof(X_train, X_test, y_train, models)

    # 6. 앙상블
    results, best_key = ensemble(oof_preds, test_preds, y_train, oof_scores)

    # 7. 제출 파일 저장
    test_ids = test.iloc[:, 0]  # 첫 번째 컬럼을 ID로 사용 (실제 ID 컬럼명으로 교체)
    save_submission(results[best_key]["test"], test_ids, best_key, results)

    print("\n✅ 파이프라인 완료!")


if __name__ == "__main__":
    main()
